# E4: Attention Fusion + Pretrained Emoji Embeddings

**Experiment**: Text-conditioned attention fusion with pretrained (frozen) emoji embeddings from TweetEval.

**Architecture**: Frozen BERT (768-d) + frozen pretrained emoji (32-d) → attention fusion → concat (800-d) → MLP → 3 classes

**Controlled-experiment contract**:
- Same train/validation/test splits as E0–E3
- Text branch receives ONLY `text_without_emoji`
- Emoji branch receives ONLY `emoji_list`
- Emoji embeddings are PRETRAINED on TweetEval and FROZEN
- Same attention architecture as E3 — only emoji initialization differs

**Prerequisite**: E2 must have been run first (provides the pretrained emoji embedding artifact).

**Runtime**: GPU (T4 recommended)

## 1. Setup

In [ ]:
# Clone repository
!rm -rf /content/SentimentAnalysis
!git clone https://github.com/Chetnapadhi/SentimentAnalysis.git /content/SentimentAnalysis
%cd /content/SentimentAnalysis
!pwd

In [ ]:
# Verify GPU
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM: {props.total_memory / 1024**3:.1f} GB')

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!python -c "import torch, transformers, pandas, sklearn; print('Environment OK')"

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## 2. Build Canonical Data Pipeline

In [ ]:
%cd /content/SentimentAnalysis

!python -m src.data.inspect_datasets
!python -m src.data.stocktwits_adapter
!python -m src.data.build_final_dataset
!python -m src.data.preprocessing

In [ ]:
# Verify canonical data
import os
import pandas as pd

base = 'data/processed/canonical'
for split in ['train', 'validation', 'test']:
    path = f'{base}/final_{split}.jsonl'
    if os.path.exists(path):
        df = pd.read_json(path, lines=True)
        print(f'{split}: {len(df)} rows')
    else:
        print(f'{split}: MISSING!')

assert os.path.exists(f'{base}/final_train.jsonl'), 'Canonical data not found!'

## 3. Load Pretrained Emoji Embedding from E2

E4 uses the same pretrained emoji embedding created during E2. Copy it from Google Drive.

In [ ]:
import os
import shutil

# Source: where E2 saved the embedding on Drive
drive_src = '/content/drive/MyDrive/SentimentAnalysis/models/emoji_embeddings/stocktwits_emoji_embedding_e2_1610x32.pt'
# Destination: where E4 expects it
local_dst = 'models/emoji_embeddings/stocktwits_emoji_embedding_e2_1610x32.pt'

os.makedirs('models/emoji_embeddings', exist_ok=True)

if os.path.exists(drive_src):
    shutil.copy2(drive_src, local_dst)
    print(f'Copied pretrained embedding from Drive')
else:
    print(f'ERROR: Pretrained embedding not found at {drive_src}')
    print('You must run E2 first and backup results to Drive.')
    raise FileNotFoundError(f'Missing: {drive_src}')

In [ ]:
# Verify the embedding artifact
import torch

artifact = torch.load(local_dst, map_location='cpu')
print(f"Embedding shape: {artifact['emoji_embedding'].shape}")
print(f"Vocab size: {artifact['vocab_size']}")
print(f"Embedding dim: {artifact['embedding_dim']}")
print(f"Pretrained count: {artifact.get('tweet_eval_pretrained_count', 'N/A')}")

assert artifact['emoji_embedding'].shape == (1610, 32), 'Wrong embedding shape!'
print('\nPretrained emoji embedding verified.')

## 4. Train E4

In [ ]:
%cd /content/SentimentAnalysis

# Run E4 training + evaluation
!RUN_E4=1 python run_e4.py

## 5. Check Results

In [ ]:
!find results/E4 -maxdepth 2 -type f | sort

In [ ]:
import json

with open('results/E4/metrics.json') as f:
    metrics = json.load(f)

print('=' * 60)
print('E4 OFFICIAL RESULTS')
print('=' * 60)
print(f"Accuracy:        {metrics['accuracy']:.6f}")
print(f"Macro Precision: {metrics['macro_precision']:.6f}")
print(f"Macro Recall:    {metrics['macro_recall']:.6f}")
print(f"Macro F1:        {metrics['macro_f1']:.6f}")
print(f"Best Epoch:      {metrics['best_epoch']}")
print(f"Best Val F1:     {metrics['best_val_macro_f1']:.6f}")

In [ ]:
from IPython.display import Image, display

display(Image('results/E4/confusion_matrix.png'))
display(Image('results/E4/training_history.png'))

## 6. Backup to Google Drive

In [ ]:
import os

drive_dir = '/content/drive/MyDrive/SentimentAnalysis/results/E4'
os.makedirs(drive_dir, exist_ok=True)

!cp -r results/E4/* {drive_dir}/

print('E4 results backed up to Drive.')
!find {drive_dir} -maxdepth 2 -type f | sort